1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰 수 초과 방지, 답변 생성 시간 단축
3. 임베딩해서 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

# 1. 문서의 내용을 읽는다

In [ ]:
%pip install python-docx

In [ ]:
from docx import Document

document = Document('./tax.docx')
print(f'document == {dir(document)}')
full_text = ''
for index, paragraph in enumerate(document.paragraphs):
    print(f'paragraph == {paragraph.text}')
    full_text += f'{paragraph.text}\n'

# 2. 문서를 쪼갠다

In [ ]:
%pip install tiktoken

In [2]:
import tiktoken 

def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model('gpt-4o')
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)

    text_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i: i+chunk_size]
        decoded = encoder.decode(chunk) # 숫자 리스트를 다시 텍스트로 변환
        text_list.append(decoded)

    return text_list

In [3]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

# 3. 문서 임베딩

In [ ]:
%pip install chromadb

In [4]:
import chromadb

chroma_client = chromadb.Client()

In [5]:
collection_name = 'tax_collection'
#tax_collection = chroma_client.create_collection(collection_name)

In [6]:
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
openai_embedding = OpenAIEmbeddingFunction(model_name='text-embedding-3-large')

In [7]:
tax_collection = chroma_client.get_or_create_collection(collection_name, embedding_function=openai_embedding)

In [8]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f'{index}')

In [11]:
tax_collection.add(documents=chunk_list, ids=id_list)

# 4. 유사도 검색

In [12]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_doc = tax_collection.query(query_texts=query)

In [ ]:
retrieved_doc['documents'][0]

# 5. LLM 질의

In [ ]:
%pip install openai

In [18]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {"role": "system", "content": f"당신은 한국의 소득세 전문가입니다. 아래 내용을 참고해서 사용자의 질문에 답변하세요 {retrieved_doc['documents'][0]}"},
        {"role": "user", "content": query}
    ]
)

In [ ]:
response.choices[0].message.content